# 05 — Modelo fundacional TTM (IBM Granite TinyTimeMixer r2)

**RESPIR-AI · TFG** — Predicción horaria de la OUR con **ibm-granite/granite-timeseries-ttm-r2**,
variante 512→96 (contexto 512 h, predicción de 96 pasos recortada a 48), cargada con
`tsfm_public.toolkit.get_model` (tsfm_public 0.3.9). Con solo **0.8 M de parámetros** tiene 142–148× menos parámetros que Chronos-2 según la variante
(805.280 parámetros en zero-shot, 842.848 con covariables; dos órdenes de magnitud), lo que permite medir también la inferencia en CPU.

Cuatro escenarios sobre las mismas 36 ventanas del protocolo v2:

- **E1 — Zero-shot**: pesos preentrenados sin adaptación, univariante.
- **E2 — FT univariante few-shot 5 %**: fine-tuning completo con el 5 % de las ventanas de train (168 de 3363), early stopping en validación (paciencia 5, máx. 50 épocas).
- **E3 — FT multivariante 5 %**: cabecera reconfigurada a 20 canales (OUR + 13 proceso + 6 meteo) con `decoder_mode="mix_channel"` y `prediction_channel_indices=[0]`.
- **E4 — FT con covariables futuras 5 % (control)**: además, las 6 meteo se declaran `exogenous_channel_indices` con `enable_forecast_channel_mixing=True`, y en inferencia se aportan sus valores futuros reales.

Semilla 42; AdamW lr=1e-4, batch 64. Al cambiar el número de canales (E3/E4) las capas afectadas
se reinicializan (`ignore_mismatched_sizes=True`) y se reaprenden durante el fine-tuning.

## 1. Carga de datos, modelo y ventanas

In [ ]:
import time
import numpy as np, pandas as pd, torch
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
from common_eval import *
torch.manual_seed(42); np.random.seed(42)
DEV = "cuda"

df, proto = load_all()
origins = get_origins(df, proto)
Y = true_targets(df, origins)

# get_model resuelve la revisión 512-96 adecuada del repositorio r2
ttm = get_model("ibm-granite/granite-timeseries-ttm-r2",
                context_length=512, prediction_length=96)
n_params_ttm = sum(p.numel() for p in ttm.parameters())
print(f"Parámetros: {n_params_ttm/1e6:.2f} M · ctx {ttm.config.context_length} → pred {ttm.config.prediction_length}")

CHANNELS = ['our'] + PROC + METEO      # canal 0 = objetivo
Xall = df[CHANNELS].to_numpy(dtype=np.float32)
ctx_uni   = torch.tensor(np.stack([Xall[o['iloc']-511:o['iloc']+1, :1] for o in origins]))
ctx_multi = torch.tensor(np.stack([Xall[o['iloc']-511:o['iloc']+1, :]  for o in origins]))

## 2. Utilidades de inferencia (GPU y CPU) y de fine-tuning

La predicción es de 96 pasos y se recorta a 48; la métrica se calcula recortando además a cada H. Los tiempos por ventana se promedian sobre 3 repeticiones.

In [ ]:
def infer(model, past, fut=None, device="cuda", n_rep=3):
    model = model.to(device).eval()
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    with torch.no_grad():
        t0 = time.perf_counter()
        for _ in range(n_rep):
            out = model(past_values=past.to(device),
                        future_values=fut.to(device) if fut is not None else None,
                        return_loss=False)
        if device == "cuda": torch.cuda.synchronize()
        t = (time.perf_counter() - t0) / n_rep / past.shape[0]
    mem = torch.cuda.max_memory_allocated()/1e6 if device == "cuda" else np.nan
    P = out.prediction_outputs[..., 0].float().cpu().numpy()   # canal objetivo
    return P[:, :48], t, mem

# Ventanas deslizantes de 512+96 h dentro de cada segmento de train/val
WIN = 512 + 96
rng = np.random.default_rng(42)
train_end_ts = pd.Timestamp(proto['particion']['train']['fin'])
val_start_ts = pd.Timestamp(proto['particion']['val']['inicio'])
val_end_ts   = pd.Timestamp(proto['particion']['val']['fin'])
def contiguous_series(mask):
    return [g for _, g in df[mask & (df.segment_id >= 0)].groupby('segment_id')]
tr_chunks = contiguous_series(df.index <= train_end_ts)
va_chunks = contiguous_series((df.index >= val_start_ts) & (df.index <= val_end_ts))
def window_starts(chunks):
    starts = []
    for g in chunks:
        i0 = df.index.get_loc(g.index[0])
        starts.extend(range(i0, i0 + len(g) - WIN + 1))
    return np.array(starts)
tr_starts_all, va_starts_all = window_starts(tr_chunks), window_starts(va_chunks)
tr_starts = rng.choice(tr_starts_all, size=int(0.05*len(tr_starts_all)), replace=False)  # few-shot 5%
va_starts = va_starts_all[::4]
print(f"{len(tr_starts)} ventanas de fine-tuning (5% de {len(tr_starts_all)}) · {len(va_starts)} de validación")

def make_tensors(starts, n_ch):
    past = np.stack([Xall[s:s+512, :n_ch] for s in starts])
    fut  = np.stack([Xall[s+512:s+608, :n_ch] for s in starts])
    return torch.tensor(past), torch.tensor(fut)

def finetune(model, n_ch, tag, max_epochs=50, patience=5, lr=1e-4, batch=64):
    torch.manual_seed(42); np.random.seed(42)
    Xtr, Ytr = make_tensors(tr_starts, n_ch)
    Xva, Yva = make_tensors(va_starts, n_ch)
    model = model.to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    best, best_state, bad = np.inf, None, 0
    torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(Xtr.shape[0])
        for k in range(0, Xtr.shape[0], batch):
            idx = perm[k:k+batch]
            out = model(past_values=Xtr[idx].to(DEV), future_values=Ytr[idx].to(DEV),
                        return_loss=True)
            opt.zero_grad(); out.loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(model(past_values=Xva.to(DEV), future_values=Yva.to(DEV),
                             return_loss=True).loss)
        if vl < best - 1e-5:
            best, bad = vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    t_ft = time.perf_counter() - t0
    model.load_state_dict(best_state)
    return model, t_ft, torch.cuda.max_memory_allocated()/1e6, ep+1

## 3. E1 — Zero-shot (GPU y CPU)

In [ ]:
ttm_rows, ttm_win, ttm_cost = [], [], []
P1, t_gpu, mem1 = infer(ttm, ctx_uni, device="cuda")
_,  t_cpu, _    = infer(ttm, ctx_uni, device="cpu")
rows, wins = evaluate_preds(P1, Y, "ttm", "E1_zs", t_gpu, 512, origins,
                            extra={"covariables": "ninguna"})
ttm_rows += rows; ttm_win += wins
ttm_cost.append({"modelo":"ttm","escenario":"E1_zs","n_parametros":n_params_ttm,
                 "t_train_s":0.0,"t_inf_ventana_gpu_s":t_gpu,"t_inf_ventana_cpu_s":t_cpu,
                 "mem_gpu_pico_MB":mem1,"dispositivo":"GPU+CPU"})
print(f"E1 ZS · MAE@48={rows[-1]['MAE']:.3f} · GPU {t_gpu*1000:.2f} ms/ventana · CPU {t_cpu*1000:.2f} ms/ventana")

## 4. E2 — Fine-tuning univariante few-shot 5 %

In [ ]:
TTM_ID = "ibm-granite/granite-timeseries-ttm-r2"
m2 = get_model(TTM_ID, context_length=512, prediction_length=96)
m2, t_ft, mem_ft, ep = finetune(m2, 1, "E2")
P2, t_g, mem_i = infer(m2, ctx_uni, device="cuda")
_,  t_c, _     = infer(m2, ctx_uni, device="cpu")
rows, wins = evaluate_preds(P2, Y, "ttm", "E2_ft_univariante_5pct", t_g, 512, origins,
                            extra={"covariables": f"ninguna (few-shot 5%, {ep} épocas)"})
ttm_rows += rows; ttm_win += wins
ttm_cost.append({"modelo":"ttm","escenario":"E2_ft_univariante_5pct","n_parametros":n_params_ttm,
                 "t_train_s":t_ft,"t_inf_ventana_gpu_s":t_g,"t_inf_ventana_cpu_s":t_c,
                 "mem_gpu_pico_MB":max(mem_ft,mem_i),"dispositivo":"GPU+CPU"})
del m2; torch.cuda.empty_cache()

## 5. E3 — Fine-tuning multivariante (channel-mixing) 5 %

Se reconfigura la cabecera a 20 canales con mezcla de canales en el decodificador; solo se predice el canal 0 (OUR).

In [ ]:
m3 = get_model(TTM_ID, context_length=512, prediction_length=96,
               num_input_channels=len(CHANNELS), decoder_mode="mix_channel",
               prediction_channel_indices=[0], ignore_mismatched_sizes=True)
m3, t_ft, mem_ft, ep = finetune(m3, len(CHANNELS), "E3")
P3, t_g, mem_i = infer(m3, ctx_multi, device="cuda")
rows, wins = evaluate_preds(P3, Y, "ttm", "E3_ft_multi_5pct", t_g, 512, origins,
                            extra={"covariables": f"proceso+meteo pasadas, mix_channel (5%, {ep} ep)"})
ttm_rows += rows; ttm_win += wins
ttm_cost.append({"modelo":"ttm","escenario":"E3_ft_multi_5pct",
                 "n_parametros":sum(p.numel() for p in m3.parameters()),
                 "t_train_s":t_ft,"t_inf_ventana_gpu_s":t_g,"t_inf_ventana_cpu_s":np.nan,
                 "mem_gpu_pico_MB":max(mem_ft,mem_i),"dispositivo":"GPU"})
del m3; torch.cuda.empty_cache()

## 6. E4 — Fine-tuning con covariables meteo futuras (control) 5 %

Las 6 variables meteorológicas se declaran exógenas conocidas a futuro (`exogenous_channel_indices` + `enable_forecast_channel_mixing`); en inferencia se aportan sus 96 h futuras reales.

In [ ]:
meteo_idx = [CHANNELS.index(c) for c in METEO]
m4 = get_model(TTM_ID, context_length=512, prediction_length=96,
               num_input_channels=len(CHANNELS), decoder_mode="mix_channel",
               prediction_channel_indices=[0], exogenous_channel_indices=meteo_idx,
               enable_forecast_channel_mixing=True, fcm_context_length=5,
               fcm_use_mixer=True, fcm_prepend_past=True, ignore_mismatched_sizes=True)
m4, t_ft, mem_ft, ep = finetune(m4, len(CHANNELS), "E4")
fut = np.zeros((len(origins), 96, len(CHANNELS)), dtype=np.float32)
for w, o in enumerate(origins):
    for ci in meteo_idx:
        fut[w, :, ci] = Xall[o['iloc']+1:o['iloc']+97, ci]   # meteo real futura
P4, t_g, mem_i = infer(m4, ctx_multi, fut=torch.tensor(fut), device="cuda")
rows, wins = evaluate_preds(P4, Y, "ttm", "E4_ft_exog_futuras_5pct", t_g, 512, origins,
                            extra={"covariables": f"meteo futuras (control), mix_channel (5%, {ep} ep)"})
ttm_rows += rows; ttm_win += wins
ttm_cost.append({"modelo":"ttm","escenario":"E4_ft_exog_futuras_5pct",
                 "n_parametros":sum(p.numel() for p in m4.parameters()),
                 "t_train_s":t_ft,"t_inf_ventana_gpu_s":t_g,"t_inf_ventana_cpu_s":np.nan,
                 "mem_gpu_pico_MB":max(mem_ft,mem_i),"dispositivo":"GPU"})
del m4; torch.cuda.empty_cache()

pd.DataFrame(ttm_rows).to_csv("results/resultados_ttm.csv", index=False)
print(pd.DataFrame(ttm_rows).pivot(index='escenario', columns='H', values='MAE').round(3))

## 7. Resultados obtenidos (ejecución real) — MAE por escenario y horizonte

| Escenario | H=6 | H=12 | H=24 | H=48 |
|---|---|---|---|---|
| E1_zs | 3.676 | 3.915 | 4.001 | 4.424 |
| E2_ft_univariante_5pct | 3.575 | 3.799 | 3.869 | 4.330 |
| E3_ft_multi_5pct | 3.613 | 3.863 | 3.914 | 4.330 |
| E4_ft_exog_futuras_5pct | 3.623 | 3.763 | 3.810 | 4.285 |

**Lectura**: TTM zero-shot (E1) ya es competitivo (MAE@48 = 4.424, similar a Chronos-2 ZS con
dos órdenes de magnitud menos parámetros (142–148× según la variante: 805.280 en zero-shot, 842.848 con covariables)). El few-shot 5 % univariante (E2) aporta una mejora
consistente en todos los horizontes (p. ej. 3.575 vs 3.676 a H=6) con solo ~1 s de entrenamiento.
El multivariante (E3) no mejora al univariante — las capas nuevas de 20 canales deben reaprenderse
con solo 168 ventanas —, mientras que **E4 (meteo futuras como control) es el mejor TTM en
H=12–48** (MAE@24 = 3.810), confirmando la utilidad del pronóstico meteorológico para el horizonte
largo, en línea con lo observado en Chronos-2 (E3/E6). La inferencia tarda ~0.3 ms/ventana tanto
en GPU como en CPU: el despliegue de TTM en la EDAR no requeriría GPU.

## Resultados guardados

Este cuaderno requiere horas de GPU para re-ejecutarse por completo; la celda siguiente carga los resultados ya calculados desde `results/` y puede ejecutarse en cualquier máquina.

In [1]:
# Resultados guardados (esta celda se puede ejecutar sin GPU)
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"common_eval.py").exists() else Path.cwd().parent
os.chdir(ROOT)
import pandas as pd
d = pd.read_csv("results/resultados_ttm.csv")
print(d.pivot(index="escenario", columns="H", values="MAE").round(3).to_string())

H                           6      12     24     48
escenario                                          
E1_zs                    3.676  3.915  4.001  4.424
E2_ft_univariante_5pct   3.575  3.799  3.869  4.330
E3_ft_multi_5pct         3.613  3.863  3.914  4.330
E4_ft_exog_futuras_5pct  3.623  3.763  3.810  4.285
